# Feature Engineering

## Learning Objectives
1. Implement polynomial features and interaction terms from scratch using NumPy.
2. Build sklearn ColumnTransformer pipelines handling numerical and categorical data together.
3. Extract time-series features (lag, rolling statistics, calendar) for temporal models.
4. Compare encoding strategies and apply automated feature selection to reduce dimensionality.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings("ignore")


## Level 1: Polynomial Features + Interaction Terms (NumPy)

In [ ]:
def polynomial_features(X, degree=2, include_bias=False):
    """
    Expand X to polynomial features up to `degree`.
    For 2 features [a, b] and degree=2: [a, b, a^2, a*b, b^2]
    """
    n_samples, n_features = X.shape
    terms = list(range(n_features)) if include_bias else list(range(n_features))
    output_cols = [X]  # include original features

    # Interaction terms: all pairs (i, j) with i <= j
    for i in range(n_features):
        for j in range(i, n_features):
            if i == j:
                output_cols.append((X[:, i] ** 2).reshape(-1, 1))  # squared
            else:
                output_cols.append((X[:, i] * X[:, j]).reshape(-1, 1))  # cross-term

    if degree >= 3:
        for i in range(n_features):
            output_cols.append((X[:, i] ** 3).reshape(-1, 1))

    result = np.hstack(output_cols)
    return result


# Verify on simple 2D data
X_demo = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
X_poly = polynomial_features(X_demo, degree=2)
print("Original shape:", X_demo.shape, "→ Polynomial shape:", X_poly.shape)
print("Original features:   ", X_demo[0])
print("Polynomial features: ", X_poly[0])
# Expected for [1,2]: [1, 2, 1, 2, 4] = original + a^2 + a*b + b^2

# Show improvement on non-linear classification
from sklearn.datasets import make_circles
X_circ, y_circ = make_circles(n_samples=400, noise=0.15, random_state=42)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
X_tr, X_te, y_tr, y_te = train_test_split(X_circ, y_circ, test_size=0.25, random_state=42)

lr_linear = LogisticRegression(random_state=42).fit(X_tr, y_tr)
X_tr_poly = polynomial_features(X_tr, degree=3)
X_te_poly = polynomial_features(X_te, degree=3)
lr_poly = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_poly, y_tr)

print(f"
Linear LR on circles: {lr_linear.score(X_te, y_te):.4f}")
print(f"Degree-3 poly LR:     {lr_poly.score(X_te_poly, y_te):.4f}")


## Level 2: ColumnTransformer + FeatureUnion Pipeline

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# Create a synthetic mixed (numerical + categorical) dataset
np.random.seed(42)
n = 800
age = np.random.randint(18, 70, n).astype(float)
income = np.random.exponential(scale=50000, size=n)
experience = np.random.randint(0, 40, n).astype(float)
job_type = np.random.choice(["engineer", "manager", "analyst", "intern"], n)
education = np.random.choice(["high_school", "bachelors", "masters", "phd"], n)

# Target: high earner (income > 60000)
y_mixed = (income > 60000).astype(int)

# Build DataFrame
df = pd.DataFrame({
    "age": age, "income_raw": income, "experience": experience,
    "job_type": job_type, "education": education,
})
# Drop target-leaking column
df_feat = df.drop(columns=["income_raw"])

num_features = ["age", "experience"]
cat_features = ["job_type", "education"]

# Numerical pipeline: scale + polynomial interactions
num_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
])

# Categorical pipeline: one-hot encode
cat_pipeline = Pipeline([
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Combine with ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features),
], remainder="drop")

# Full pipeline: preprocessing + classifier
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=500, C=1.0, random_state=42)),
])

from sklearn.model_selection import cross_val_score
scores = cross_val_score(full_pipeline, df_feat, y_mixed, cv=5, scoring="roc_auc")
print(f"Pipeline ROC-AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

full_pipeline.fit(df_feat, y_mixed)
print(f"Train accuracy: {full_pipeline.score(df_feat, y_mixed):.4f}")
print(f"Transformed feature shape: {preprocessor.fit_transform(df_feat).shape}")


## Real-World Example 1: Time Series Feature Extraction

In [ ]:
# Time series features: lags, rolling stats, calendar indicators
# These convert a raw time series into a flat feature matrix for tabular models.

np.random.seed(42)
dates = pd.date_range("2022-01-01", periods=365, freq="D")
sales = (100
         + 20 * np.sin(2 * np.pi * np.arange(365) / 365)  # yearly seasonality
         + 5 * np.sin(2 * np.pi * np.arange(365) / 7)     # weekly seasonality
         + np.random.normal(0, 5, 365))                    # noise
df_ts = pd.DataFrame({"date": dates, "sales": sales})
df_ts.set_index("date", inplace=True)


def extract_time_features(df, target_col="sales", lag_days=(1, 7, 14, 28),
                          rolling_windows=(7, 14, 28)):
    """Create lag, rolling, and calendar features from a daily time series."""
    df_out = df.copy()

    # Lag features: value N days ago
    for lag in lag_days:
        df_out[f"lag_{lag}d"] = df_out[target_col].shift(lag)

    # Rolling statistics: capture local trend and volatility
    for win in rolling_windows:
        df_out[f"roll_mean_{win}d"] = df_out[target_col].shift(1).rolling(win).mean()
        df_out[f"roll_std_{win}d"] = df_out[target_col].shift(1).rolling(win).std()

    # Calendar features: periodicity information
    df_out["day_of_week"] = df_out.index.dayofweek         # 0=Monday
    df_out["day_of_month"] = df_out.index.day
    df_out["month"] = df_out.index.month
    df_out["is_weekend"] = (df_out.index.dayofweek >= 5).astype(int)
    df_out["week_of_year"] = df_out.index.isocalendar().week.astype(int)

    df_out = df_out.dropna()  # drop rows with NaN from lags/rolling
    return df_out


df_features = extract_time_features(df_ts)
print("Original shape:", df_ts.shape)
print("After feature extraction:", df_features.shape)
print("Features created:", [c for c in df_features.columns if c != "sales"])

# Quick regression test
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_ts = df_features.drop(columns=["sales"]).values
y_ts = df_features["sales"].values
split = int(len(X_ts) * 0.8)
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_ts[:split], y_ts[:split])
mae = mean_absolute_error(y_ts[split:], rf_reg.predict(X_ts[split:]))
print(f"RF regression MAE on test period: {mae:.2f} sales units")


## Real-World Example 2: Categorical Encoding Comparison

In [ ]:
# Ordinal vs One-Hot vs Target Encoding (with cross-fitting to prevent target leakage)

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# Synthetic dataset with a high-cardinality categorical feature
np.random.seed(42)
n_samples = 1000
n_categories = 50
categories = [f"cat_{i:02d}" for i in range(n_categories)]
X_cat_raw = np.random.choice(categories, n_samples)

# Some categories are genuinely predictive
cat_effect = {c: np.random.randn() for c in categories}
noise = np.random.normal(0, 0.5, n_samples)
y_cat_cont = np.array([cat_effect[c] for c in X_cat_raw]) + noise
y_cat = (y_cat_cont > 0).astype(int)

df_cat = pd.DataFrame({"category": X_cat_raw})

# Ordinal encoding: fast, compact, but implies ordering
ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_ord = ord_enc.fit_transform(df_cat)

# One-hot encoding: sparse, no ordering, bad for high-cardinality
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_ohe = ohe.fit_transform(df_cat)

print("Encoding sizes:")
print(f"  Ordinal:  {X_ord.shape}  (1 column per feature)")
print(f"  One-Hot:  {X_ohe.shape}  ({n_categories} columns for {n_categories} categories)")

# Target encoding: mean of target per category (risk of leakage without CV)
def target_encode_cv(X_col, y, n_splits=5, smoothing=10):
    """Cross-fitted target encoding to prevent target leakage."""
    encoded = np.zeros(len(y))
    global_mean = y.mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for tr_idx, val_idx in kf.split(X_col):
        counts = {}
        sums = {}
        for i in tr_idx:
            c = X_col[i]
            counts[c] = counts.get(c, 0) + 1
            sums[c] = sums.get(c, 0) + y[i]
        for i in val_idx:
            c = X_col[i]
            n_c = counts.get(c, 0)
            s_c = sums.get(c, global_mean * n_c)
            # Smoothed target mean: shrinks toward global mean for rare categories
            encoded[i] = (s_c + smoothing * global_mean) / (n_c + smoothing)
    return encoded.reshape(-1, 1)

X_te_enc = target_encode_cv(X_cat_raw, y_cat)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

for name, X_enc in [("Ordinal", X_ord), ("One-Hot", X_ohe), ("Target (CV)", X_te_enc)]:
    pipe = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=300, random_state=42))])
    scores = cross_val_score(pipe, X_enc, y_cat, cv=5, scoring="roc_auc")
    print(f"{name:<18} ROC-AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")


## Real-World Example 3: Feature Selection Pipeline

In [ ]:
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, mutual_info_classif, RFECV
)
from sklearn.ensemble import RandomForestClassifier as RF

np.random.seed(42)
# High-dimensional dataset: many irrelevant/low-variance features
X_hi, y_hi = make_classification(
    n_samples=800, n_features=50, n_informative=8,
    n_redundant=10, n_repeated=5, random_state=42
)

print(f"Starting feature count: {X_hi.shape[1]}")

# Step 1: Remove near-zero-variance features
vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X_hi)
print(f"After VarianceThreshold:  {X_vt.shape[1]} features")

# Step 2: Mutual information filter (univariate)
selector_mi = SelectKBest(mutual_info_classif, k=20)
X_mi = selector_mi.fit_transform(X_vt, y_hi)
print(f"After SelectKBest (MI):   {X_mi.shape[1]} features")

# Step 3: Recursive Feature Elimination with CV
rfecv = RFECV(
    estimator=RF(n_estimators=50, random_state=42, n_jobs=-1),
    step=1,
    cv=5,
    scoring="roc_auc",
    min_features_to_select=3,
    n_jobs=-1,
)
rfecv.fit(X_mi, y_hi)
X_rfe = rfecv.transform(X_mi)
print(f"After RFECV:              {X_rfe.shape[1]} features (optimal per CV)")

# Full selection pipeline
selection_pipeline = Pipeline([
    ("vt", VarianceThreshold(threshold=0.01)),
    ("mi", SelectKBest(mutual_info_classif, k=20)),
    ("rfecv", RFECV(RF(n_estimators=50, random_state=42, n_jobs=-1), cv=5, scoring="roc_auc", n_jobs=-1)),
    ("clf", RF(n_estimators=100, random_state=42, n_jobs=-1)),
])

from sklearn.model_selection import cross_val_score
scores_full = cross_val_score(RF(n_estimators=100, random_state=42), X_hi, y_hi, cv=5, scoring="roc_auc")
print(f"
All 50 features ROC-AUC:   {scores_full.mean():.4f}")

scores_sel = cross_val_score(RF(n_estimators=100, random_state=42), X_rfe, y_hi, cv=5, scoring="roc_auc")
print(f"Selected features ROC-AUC: {scores_sel.mean():.4f}")
print(f"Reduction: {X_hi.shape[1]} → {X_rfe.shape[1]} features  ({100*(1-X_rfe.shape[1]/X_hi.shape[1]):.0f}% fewer)")
